# Task 2 — Mini-Factorio environment, harness, and baseline evaluation

**Assignment (per `plan.md`):** build a measurable simplified Factorio-like environment where a cheap baseline LLM proposes floorplan edits to maximize green-science-pack production. Show the baseline is not already optimal.

**What this notebook does:**
1. Documents the environment, simulator, reward, harness, and design choices.
2. Loads the validation split of layouts.
3. Loads the baseline model `Qwen2.5-Coder-1.5B-Instruct`.
4. Runs the baseline on 20 val layouts × 5 samples each.
5. Compares baseline rewards against a handcrafted layout (evidence baseline is not optimal).
6. Saves results to `results/baseline_eval.json` for later comparison against GRPO checkpoints.

**Reader path:** Morgan or any evaluator can read Sections 1–6 without running any code and see the full design story. Sections 7–12 are the executable eval.

## 1. What we built

A three-part package:

- **`mini_factorio/`** — the environment. Layout schema, real Factorio 2.0.77 recipes/entities pulled from the [factoriolab](https://github.com/factoriolab/factoriolab) JSON dataset (MIT licensed, same data underlying factoriolab.dev). A static rate-based simulator that returns steady-state green-science pack throughput given a validated layout. A composite reward function.
- **`harness/`** — everything that connects the LLM to the environment. Prompt builder, edit schema (Pydantic), tolerant JSON parser, edit applier with per-edit rollback, end-to-end evaluator.
- **`translator/`** — Mini-Factorio Layout → Factorio Lua commands (for FLE validation) and → Factorio blueprint dict (for portability). Inserts a substation + creative-mode power source since our sim skips electricity.

### Environment specification

| Item | Value | Source |
|---|---|---|
| Grid | 16×16 tiles | Fits full green-science chain; 20×20 available if tight |
| Target | maximize `logistic-science-pack`/sec | Real Factorio recipe = 1 inserter + 1 transport-belt (6s) |
| Machines | `electric-mining-drill` (3×3), `stone-furnace` (2×2), `assembling-machine-1` (3×3) | factoriolab sizes |
| Placeables | `transport-belt` (1×1, 15 items/sec), `inserter` (1×1, 0.83 items/sec) | Real belt speed; documented inserter constant |
| Resources | iron-ore, copper-ore, coal, stone (mineable patches) | Coal needed as furnace fuel |
| Budget | starting pool of iron-plate, copper-plate, stone, gears, circuits | Used for materials tracking (secondary reward term) |

### Recipe chain

```
iron-ore --(furnace + coal)--> iron-plate
copper-ore --(furnace + coal)--> copper-plate
iron-plate ×2 --(assembler)--> iron-gear-wheel
copper-plate --(assembler)--> copper-cable ×2
iron-plate + copper-cable ×3 --(assembler)--> electronic-circuit
iron-plate + iron-gear-wheel --(assembler)--> transport-belt ×2
electronic-circuit + iron-gear-wheel + iron-plate --(assembler)--> inserter
inserter + transport-belt --(assembler)--> logistic-science-pack   ← green science
```

Every recipe time, ingredient count, and machine spec is generated by `mini_factorio/import_recipes.py` from the factoriolab JSON. Nothing hand-transcribed.

## 2. Simulator design

**Chosen approach: static rate-based DAG solver with fixpoint iteration.**

Real Factorio uses a tick-based simulation where individual items travel on belts. We chose to skip ticks and solve steady-state rates analytically. The reasoning:

- **Speed matters.** GRPO training needs to score hundreds of layouts per training step. A tick-based sim would be far too slow.
- **DAGs have closed-form flows.** Our recipe chain is acyclic. In steady state each node's throughput is fully determined by supply and capacity — no item-movement simulation needed.
- **FLE is the ground truth.** Our sim is validated against FLE (real Factorio via RCON) — plan.md §FLE integration. If numbers agree, the sim is trustworthy for training.

### Flow graph

Nodes = machines + inserters + belts. Edges follow item flow direction:
- **Machine → inserter** if the inserter's pickup tile is adjacent to a machine footprint tile.
- **Belt → inserter** if the inserter's pickup tile is a belt tile.
- **Inserter → machine** or **inserter → belt** symmetrically on the drop side.

**Inserter direction convention:** `direction` = drop direction. An `east`-facing inserter at `(x,y)` picks up from `(x-1, y)` and drops at `(x+1, y)`.

### Belt FCFS allocation

A single belt can carry a bounded flow (15 items/sec, yellow belt). When multiple producer inserters feed the same belt and/or multiple consumer inserters pull from it:

```
effective_flow = min(sum_producer_nominal, sum_consumer_demand, BELT_SPEED)
```

The flow is distributed **first-come-first-served by upstream position** on the belt — upstream producers succeed first, upstream consumers get first pick. This matches real Factorio's behavior on a shared bus (an inserter can't take more than what has reached its tile). Test coverage for this is in `mini_factorio/tests.py`.

### Fixpoint iteration

Because belts create feedback (producer supply depends on consumer demand, consumer effective rate depends on producer supply), we iterate: initialize machine rates to nominal, then repeatedly recompute inserter throughputs, belt flows, and machine rates constrained by input supply. Rates only decrease across iterations, so convergence is guaranteed. In practice we converge in a handful of iterations (tolerance `1e-9`, max 200).

### Furnace fuel handling

Stone furnaces (90 kW) require coal (4 MJ/unit) as an additional input at rate `0.0225 coal/sec` per fully-active furnace. Without a coal supply, the furnace outputs zero. This is modelled as an additional required-input rate in the machine's supply-ratio computation.

## 3. Reward formula

```
R = green_science_rate  −  α · materials  −  β · cells  −  γ · machines
```

| Term | Symbol | Default | Meaning |
|---|---|---|---|
| Green-science rate (items/sec) | — | 1 (primary) | The main deliverable — from the simulator |
| Materials used | α | 0.001 | Total construction cost (real factoriolab recipes) — tie-breaker |
| Total cells occupied | β | 0.01 | Grid footprint — penalizes sprawl |
| Machine count | γ | 0.05 | Number of placed machines + inserters — penalizes over-building |

Weights are deliberately small so the science rate dominates. Secondary terms only matter when comparing layouts with equal science output.

**Weight tuning discipline:** per `plan.md` §Reward tuning, weights are tuned only on the 60-layout training split. The 20 validation layouts are untouched until the final GRPO evaluation. This prevents overfitting reward-shape choices to the val split.

**Per-checkpoint reporting:** the composite goes to GRPO as a single scalar. All raw components (green science rate, materials, cells, machine count, valid-outputs %) are logged separately per checkpoint so the writeup can present them alongside the composite — plan.md §Reward reporting.

## 4. Harness design

The harness is the code that makes model outputs testable:

```
layout → prompt → model completion → parsed edits → applied layout → reward
```

### Prompt (`harness/prompt_builder.py`)

Four sections:
1. **Rules block** — recipe database, machine specs, belt/inserter constants. Static across all prompts (same environment).
2. **Edit schema block** — the six edit operations the model may emit, with JSON grammar.
3. **Current layout block** — the layout JSON, indented for readability.
4. **Instruction** — "return only the edits JSON".

Chat-format version wraps this with a system prompt for Qwen's chat template.

### Edit schema (`harness/edit_schema.py`)

Six operations, discriminated by `op`:

```
add_entity      { id, type, x, y, recipe?, target_resource? }
remove_entity   { id }
add_inserter    { id, x, y, direction }
add_belt        { id, item, tiles: [[x, y, dir], ...] }
remove_belt     { id }
extend_belt     { id, tiles: [[x, y, dir], ...] }
```

Pydantic gives us free validation and JSON I/O.

### Parser (`harness/edit_parser.py`) — deliberately tolerant

Real LLM completions often wrap JSON in ```json fences or add prose. The parser:
1. Strips code fences if present.
2. Tries JSON parse on the stripped body.
3. On failure, hunts for the first balanced `{...}` substring and tries that.
4. Tolerates bare-list JSON (wraps into `{"edits": [...]}`).
5. Returns `parse_ok=False` if nothing validates, so the harness can dock a −1.0 penalty (plan.md §Reward wrapper).

### Applier (`harness/edit_applier.py`) — per-edit rollback

Each edit is applied against the running layout and validated in isolation. Failing edits are rolled back with a clear error string; subsequent edits still run. This matches plan.md §Edit schema: "On failure, that specific edit is rejected with a clear error string; other edits in the list still apply."

### Evaluator (`harness/evaluator.py`)

`evaluate_policy(policy, layouts, samples_per_layout)` runs the full loop end-to-end and returns an `EvalReport` with per-layout rewards, valid-edit rate, and invalid-JSON rate.

## 5. Baseline model choice

**Chosen: `Qwen2.5-Coder-1.5B-Instruct`.**

Requirements per plan.md:
- Cheap enough to fit on Colab Pro T4 with LoRA (4GB weights headroom for GRPO gradients).
- Code/JSON-friendly, since the harness expects strict JSON output.
- Small enough that the baseline is not already saturated — leaves headroom for GRPO to demonstrate improvement.

Qwen2.5-Coder-1.5B satisfies all three:
- 1.5B parameters, fits in 3GB fp16.
- Code-tuned; handles JSON schemas well.
- Small model, likely to make mistakes on a specialized factory-design task → good headroom for GRPO to prove the point.

**Not chosen:** DeepSeek-Coder-7B (too big for T4 without heavy quantization); base non-coder Qwen (worse at JSON); GPT-4 (not fine-tunable and defeats the point).

## 6. Simplifications and why

| Simplification | Reason | Impact |
|---|---|---|
| Electricity subsystem skipped (miners/assemblers assumed always powered) | ~5 new entity types would multiply schema complexity without changing green-science optimization strategy | Furnaces still need coal (per-machine input, not network); FLE translator inserts a substation + EEI power source at build time |
| Inserter throughput = 0.83 items/sec (fixed) | factoriolab's `inserter.speed` field is a game-internal rotation-speed value not directly interpretable as items/sec | 0.83 is the widely-cited vanilla yellow-inserter figure; may be revised after FLE cross-check |
| Machine tile abstraction (any footprint tile can be pickup/drop) | Real Factorio distinguishes assembler input/output tiles by side; modelling this adds complexity | Layouts remain physically buildable in FLE; slight over-permissiveness for our simulator |
| Miners don't back-pressure | Sim assumes miners always run at nominal, wasting ore if no consumer | Doesn't affect the reported green-science rate; may affect materials cost negligibly |
| No fluids, no biters, no modules, no belt tiers beyond yellow | Out of scope for green science | Documented in the report; FLE cross-check would surface any discrepancy |
| Belt routing not geometrically validated | Belts are declared as lists of tile positions; we don't enforce that they form a walkable line to their targets | Layout validator checks contiguity and non-overlap; FLE build catches remaining edge cases |

## 7. Imports and setup

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # so imports work when running from notebooks/

import json
from statistics import mean, stdev

import matplotlib.pyplot as plt

from mini_factorio.random_layouts import train_val_split
from mini_factorio.reward import compute_reward
from mini_factorio.layout import Layout, Machine, Inserter, Resource
from harness.evaluator import evaluate_policy
from harness.prompt_builder import build_chat_messages, build_prompt

## 8. Load the validation split

Seeds 1000..1019, deterministic. The 60-layout training split is separate and untouched here — plan.md §Val split discipline.

In [ ]:
_, val_layouts = train_val_split(n_train=60, n_val=20)
print(f'val split: {len(val_layouts)} layouts')
for i, lay in enumerate(val_layouts[:3]):
    print(f'layout {i}: {len(lay.machines)} machines, {len(lay.inserters)} inserters, {len(lay.belts)} belts')

## 9. Load `Qwen2.5-Coder-1.5B-Instruct`

On M2 CPU this is slow (~30 min for the full eval). On CUDA / MPS / Colab T4 it's fast (~2 min).

Requires the `[llm]` extra: `uv sync --extra llm` or `pip install -e '.[llm]'`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device != 'cpu' else torch.float32,
).to(device)
model.eval()
print(f'model loaded: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params')

In [ ]:
@torch.no_grad()
def qwen_policy(prompt: str, *, temperature: float = 0.7, max_new_tokens: int = 512) -> str:
    """Wrap the user prompt in Qwen's chat template and return the completion text."""
    messages = [
        {'role': 'system', 'content': 'You are a Factorio factory-design assistant. Output only JSON edits.'},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=temperature > 0,
        pad_token_id=tokenizer.eos_token_id,
    )
    completion = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return completion

## 10. Run the baseline

5 samples per layout × 20 layouts = 100 completions. Reduce `SAMPLES` or `N_VAL_TO_USE` for a quick smoke test.

In [ ]:
SAMPLES = 5
N_VAL_TO_USE = 20  # set smaller for a quick smoke run

report = evaluate_policy(qwen_policy, val_layouts[:N_VAL_TO_USE], samples_per_layout=SAMPLES)
print(json.dumps(report.summary(), indent=2))

## 11. Per-layout distribution

Box plot of composite reward across the 5 samples per layout. Wide boxes mean the model is inconsistent; boxes near zero or negative mean the baseline mostly produces invalid or unhelpful edits.

In [ ]:
per_layout_rewards = []
for i in range(0, len(report.episodes), SAMPLES):
    group = report.episodes[i:i + SAMPLES]
    per_layout_rewards.append([e.reward.composite for e in group])

fig, ax = plt.subplots(figsize=(10, 4))
ax.boxplot(per_layout_rewards, labels=[f'L{i}' for i in range(len(per_layout_rewards))])
ax.set_xlabel('validation layout')
ax.set_ylabel('composite reward')
ax.set_title('Baseline reward per layout (5 samples each)')
ax.axhline(0, color='gray', linewidth=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 12. Handcrafted-vs-baseline comparison

Design a small hand-crafted layout that we know produces meaningful output, then check whether its composite reward exceeds the baseline mean. If handcrafted > baseline, the baseline is not already optimal — the claim plan.md §Task 2 makes.

The handcrafted layout below is a minimal iron-plate producer (still zero green-science, but positive throughput on the chain). A richer handcrafted layout with a full green-science chain can be swapped in as we tune.

In [ ]:
def handcrafted_iron_plate_only() -> Layout:
    return Layout(
        grid_size=(16, 16),
        resources=[
            Resource(type='iron-ore', x=5, y=1, size=3),
            Resource(type='coal', x=9, y=4, size=3),
        ],
        machines=[
            Machine(id='mi', type='electric-mining-drill', x=5, y=1, target_resource='iron-ore'),
            Machine(id='mc', type='electric-mining-drill', x=9, y=4, target_resource='coal'),
            Machine(id='f',  type='stone-furnace',         x=9, y=1, recipe='iron-plate'),
        ],
        inserters=[
            Inserter(id='i1', x=8, y=1, direction='east'),
            Inserter(id='ic', x=9, y=3, direction='north'),
        ],
    )

handcrafted = handcrafted_iron_plate_only()
print('handcrafted validation errors:', handcrafted.validate_layout())
print('handcrafted reward:', compute_reward(handcrafted).to_dict())

In [ ]:
baseline_mean = report.mean_reward()
handcrafted_reward = compute_reward(handcrafted).composite
print(f'baseline mean composite: {baseline_mean:.4f}')
print(f'handcrafted composite  : {handcrafted_reward:.4f}')
print(f'evidence baseline is not optimal: {handcrafted_reward > baseline_mean}')

## 13. Interpretation

*Fill in after running:*

- **Invalid-JSON rate:** what fraction of the model's completions failed to parse as our edit schema. High rate → prompt engineering / larger model needed. Baseline eval target ≥ 70% valid outputs (plan.md §Verification).
- **Valid-edit rate:** of edits that parsed, what fraction actually applied (didn't fail validation). Low rate → the model is proposing edits that violate bounds/collisions/schema.
- **Mean composite reward vs handcrafted:** if handcrafted > baseline mean, the baseline is not automatically optimal — GRPO has room to improve. This is the plan §Task 2 claim: "Show the baseline is not already optimal."
- **Per-layout variance:** wide boxes indicate the model behaves inconsistently on the same starting layout — an argument for group-relative training (GRPO) that leverages this variance.

## 14. Save results

Persist to `results/baseline_eval.json` so the Task 3 notebook can load and compare against GRPO checkpoints.

In [ ]:
import pathlib
out_dir = pathlib.Path('../results')
out_dir.mkdir(exist_ok=True)

results = {
    'model': MODEL_ID,
    'samples_per_layout': SAMPLES,
    'n_layouts': N_VAL_TO_USE,
    'summary': report.summary(),
    'per_layout_rewards': per_layout_rewards,
    'handcrafted_reward': handcrafted_reward,
}
with open(out_dir / 'baseline_eval.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'wrote {out_dir / "baseline_eval.json"}')

## 15. Pointers

- **Full plan:** `plan.md` — every locked design decision, the DeepSeekMath GRPO objective (Eq 3, Eq 4), hyperparameters, FLE integration spec, and success criteria.
- **Sutton & Barto Task 1 demo:** `notebooks/task1_policy_iteration.ipynb` — exact policy iteration on a gridworld MDP (theorem baseline for the whole project).
- **GRPO training (Task 3):** `notebooks/task3_grpo_training.ipynb` (to be written) — runs on Colab Pro T4 using TRL `GRPOTrainer` + LoRA on Qwen2.5-Coder-1.5B.
- **FLE cross-check:** `translator/to_fle.py` translates layouts to Lua/blueprint form for validation against a real Factorio server via RCON.
- **Writeup:** `writeup/report.md` (to be written) — top-level Morgan-facing summary.